In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# =========================
# CONFIG
# =========================
BASE_URL = "https://id.jobstreet.com/id/jobs?page="
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}
MAX_PAGES = 50
DELAY_LIST_PAGE = 2
DELAY_DETAIL_PAGE = 1

job_list = []

# =========================
# FUNCTION: GET DETAIL JOB
# =========================
def get_job_details(job_url):
    try:
        response = requests.get(job_url, headers=HEADERS)
        if response.status_code != 200:
            return "", "", "", ""

        soup = BeautifulSoup(response.content, "html.parser")

        # =========================
        # 🔹 REQUIREMENTS + QUALIFICATIONS
        # =========================
        combined_requirements = []
        job_ad = soup.find("div", {"data-automation": "jobAdDetails"})

        if job_ad:
            for li in job_ad.find_all("li"):
                text = li.get_text(strip=True)
                if text:
                    combined_requirements.append(text)

        requirements = "; ".join(combined_requirements)

        # =========================
        # 🔹 SALARY
        # =========================
        salary = ""
        salary_tag = soup.find("span", {"data-automation": "job-detail-salary"})
        if salary_tag:
            salary = salary_tag.get_text(strip=True)
            salary = salary.replace("\xa0", " ")

        # =========================
        # 🔹 WORK TYPE (Full time / Part time)
        # =========================
        work_type = ""
        type_tag = soup.find("span", {"data-automation": "job-detail-work-type"})
        if type_tag:
            a_tag = type_tag.find("a")
            if a_tag:
                work_type = a_tag.get_text(strip=True)

        # =========================
        # 🔹 COMPANY NAME
        # =========================
        company = ""
        company_tag = soup.find("span", {"data-automation": "advertiser-name"})
        if company_tag:
            company = company_tag.get_text(strip=True)

        return requirements, salary, work_type, company

    except Exception as e:
        print(f"❌ Error detail: {e}")
        return "", "", "", ""

# =========================
# MAIN SCRAPING
# =========================
for page in range(1, MAX_PAGES + 1):
    print(f"\n🔄 Scraping halaman {page}...")

    try:
        response = requests.get(BASE_URL + str(page), headers=HEADERS)
        if response.status_code != 200:
            print(f"❌ Gagal akses halaman {page}")
            break

        soup = BeautifulSoup(response.content, "html.parser")

        job_cards = soup.find_all("a", {"data-automation": "jobTitle"})

        for job_card in job_cards:
            try:
                title = job_card.get_text(strip=True)
                link = "https://id.jobstreet.com" + job_card["href"]

                # 🔹 LOCATION
                location_tag = job_card.find_next("a", {"data-automation": "jobLocation"})
                location = location_tag.get_text(strip=True) if location_tag else ""

                # 🔹 DETAIL
                requirements, salary, work_type, company = get_job_details(link)

                job_list.append({
                    "Posisi": title,
                    "Perusahaan": company,
                    "Lokasi": location,
                    "Type": work_type,
                    "Gaji": salary,
                    "Requirements": requirements,
                    "Link": link
                })

                print(f"✔ {title}")

                time.sleep(DELAY_DETAIL_PAGE)

            except Exception as e:
                print(f"❌ Error job card: {e}")

        time.sleep(DELAY_LIST_PAGE)

    except Exception as e:
        print(f"❌ Error halaman: {e}")
        break

# =========================
# SAVE TO CSV
# =========================
df = pd.DataFrame(job_list)

# hapus duplikat
df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

df.to_csv("dataset.csv", index=False)

print("\n✅ Selesai! Data disimpan ke dataset.csv")



🔄 Scraping halaman 1...
✔ Manager - IFS - Entrepreneurial and Private Business (Business Development)
✔ Region Merchant Partnership & Activation Manager
✔ Supply Strategy Senior Manager
✔ Commis I25137114
✔ Fleet Management Staff
✔ Sales (live Streaming)
✔ Commis I - Banquet25086176
✔ General Manager - Sheraton Senggigi Beach Resort25069446
✔ Receptionist (BSD City)
✔ (JV2511003) Account Officer (Sales) - Jakarta
✔ Sales and Insurance Intern
✔ Assistant Manager Event Booking Center25149743
✔ Marketing Manager25149858
✔ Director Sales & Marketing25081990
✔ Service Technician [JABODETABEK]
✔ Business Representative - Manado
✔ (JV2508003) Regional Sales Head West
✔ Guest Services Agent25078929
✔ Guest Experience Agent25078931
✔ [G04] Cloud Engineer
✔ (JV2510005) Account Representative (Sales) - Samarinda
✔ Senior Business Operations Associate Manager - Flight
✔ Floor Leader
✔ (JV2510003) Account Representative (Sales) - Makassar
✔ Beauty Advisor (Kalibata City)
✔ TRANSLATION MANDARIN
✔ A

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# =========================
# CONFIG
# =========================
BASE_URL = "https://id.jobstreet.com/id/jobs?page="
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

START_PAGE = 51   # lanjut dari sini
MAX_PAGES = 100   # target halaman akhir

DELAY_LIST_PAGE = 2
DELAY_DETAIL_PAGE = 1

FILE_NAME = "dataset.csv"

job_list = []

# =========================
# FUNCTION: GET DETAIL JOB
# =========================
def get_job_details(job_url):
    try:
        response = requests.get(job_url, headers=HEADERS)
        if response.status_code != 200:
            return "", "", "", ""

        soup = BeautifulSoup(response.content, "html.parser")

        # 🔹 REQUIREMENTS
        combined_requirements = []
        job_ad = soup.find("div", {"data-automation": "jobAdDetails"})

        if job_ad:
            for li in job_ad.find_all("li"):
                text = li.get_text(strip=True)
                if text:
                    combined_requirements.append(text)

        requirements = "; ".join(combined_requirements)

        # 🔹 SALARY
        salary = ""
        salary_tag = soup.find("span", {"data-automation": "job-detail-salary"})
        if salary_tag:
            salary = salary_tag.get_text(strip=True).replace("\xa0", " ")

        # 🔹 WORK TYPE
        work_type = ""
        type_tag = soup.find("span", {"data-automation": "job-detail-work-type"})
        if type_tag:
            a_tag = type_tag.find("a")
            if a_tag:
                work_type = a_tag.get_text(strip=True)

        # 🔹 COMPANY
        company = ""
        company_tag = soup.find("span", {"data-automation": "advertiser-name"})
        if company_tag:
            company = company_tag.get_text(strip=True)

        return requirements, salary, work_type, company

    except Exception as e:
        print(f"❌ Error detail: {e}")
        return "", "", "", ""

# =========================
# MAIN SCRAPING
# =========================
for page in range(START_PAGE, MAX_PAGES + 1):
    print(f"\n🔄 Scraping halaman {page}...")

    try:
        response = requests.get(BASE_URL + str(page), headers=HEADERS)
        if response.status_code != 200:
            print(f"❌ Gagal akses halaman {page}")
            break

        soup = BeautifulSoup(response.content, "html.parser")

        job_cards = soup.find_all("a", {"data-automation": "jobTitle"})

        for job_card in job_cards:
            try:
                title = job_card.get_text(strip=True)
                link = "https://id.jobstreet.com" + job_card["href"]

                # 🔹 LOCATION
                location_tag = job_card.find_next("a", {"data-automation": "jobLocation"})
                location = location_tag.get_text(strip=True) if location_tag else ""

                # 🔹 DETAIL
                requirements, salary, work_type, company = get_job_details(link)

                job_list.append({
                    "Posisi": title,
                    "Perusahaan": company,
                    "Lokasi": location,
                    "Type": work_type,
                    "Gaji": salary,
                    "Requirements": requirements,
                    "Link": link
                })

                print(f"✔ {title}")

                time.sleep(DELAY_DETAIL_PAGE)

            except Exception as e:
                print(f"❌ Error job card: {e}")

        time.sleep(DELAY_LIST_PAGE)

    except Exception as e:
        print(f"❌ Error halaman: {e}")
        break

# =========================
# SAVE DATA (APPEND MODE)
# =========================
df = pd.DataFrame(job_list)

# hapus duplikat dari batch baru
df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

if os.path.exists(FILE_NAME):
    # load data lama
    old_df = pd.read_csv(FILE_NAME)

    # gabungkan
    df = pd.concat([old_df, df], ignore_index=True)

    # hapus duplikat global
    df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

# simpan ulang (clean version)
df.to_csv(FILE_NAME, index=False)

print("\n✅ Selesai! Data berhasil di-update tanpa duplikasi.")


🔄 Scraping halaman 51...
✔ Area Sales Manager - East Indonesia
✔ Senior Staff Finance, Accounting & Tax
✔ General Affair Officer Jakarta
✔ Finance Accounting & Tax
✔ HRIS Staff
✔ Manajemen Representatif (MR)
✔ GA & IT MANAGER
✔ Key Account Supervisor
✔ MERCHANDISER MANAGER
✔ Staff - Mekanik Mesin
✔ Guru Matematika SMA
✔ Sales Admin & Analyst
✔ Koordinator / Kepala Teknisi
✔ Teknisi
✔ Drafter Piping
✔ System Administrator
✔ Key Account Executive
✔ Business Development
✔ Perawat Kamar Operasi
✔ Electrical Engineering (Kosmetik & Farmasi)
✔ Head of Department HRD
✔ Key Account Administrative / Admin Modern Trade
✔ Driver
✔ Account Executive
✔ Operation Area FEP Sr Specialist
✔ Officer AP
✔ CS IMPORT OPERATION STAFF
✔ DIGITAL MARKETING
✔ Butcher
✔ Coordinator Area -  Consumer Food Products (Distributor Channel)
✔ Asset Management Specialist (DATA CENTER OPERATIONS)
✔ Senior Accounting

🔄 Scraping halaman 52...
✔ Social Media & Content Supervisor
✔ PAC Engineer Data Center
✔ Talent Acquisi

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# =========================
# CONFIG
# =========================
BASE_URL = "https://id.jobstreet.com/id/jobs?page="
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

START_PAGE = 66   # lanjut dari sini
MAX_PAGES = 100   # target halaman akhir

DELAY_LIST_PAGE = 2
DELAY_DETAIL_PAGE = 1

FILE_NAME = "dataset.csv"

job_list = []

# =========================
# FUNCTION: GET DETAIL JOB
# =========================
def get_job_details(job_url):
    try:
        response = requests.get(job_url, headers=HEADERS)
        if response.status_code != 200:
            return "", "", "", ""

        soup = BeautifulSoup(response.content, "html.parser")

        # 🔹 REQUIREMENTS
        combined_requirements = []
        job_ad = soup.find("div", {"data-automation": "jobAdDetails"})

        if job_ad:
            for li in job_ad.find_all("li"):
                text = li.get_text(strip=True)
                if text:
                    combined_requirements.append(text)

        requirements = "; ".join(combined_requirements)

        # 🔹 SALARY
        salary = ""
        salary_tag = soup.find("span", {"data-automation": "job-detail-salary"})
        if salary_tag:
            salary = salary_tag.get_text(strip=True).replace("\xa0", " ")

        # 🔹 WORK TYPE
        work_type = ""
        type_tag = soup.find("span", {"data-automation": "job-detail-work-type"})
        if type_tag:
            a_tag = type_tag.find("a")
            if a_tag:
                work_type = a_tag.get_text(strip=True)

        # 🔹 COMPANY
        company = ""
        company_tag = soup.find("span", {"data-automation": "advertiser-name"})
        if company_tag:
            company = company_tag.get_text(strip=True)

        return requirements, salary, work_type, company

    except Exception as e:
        print(f"❌ Error detail: {e}")
        return "", "", "", ""

# =========================
# MAIN SCRAPING
# =========================
for page in range(START_PAGE, MAX_PAGES + 1):
    print(f"\n🔄 Scraping halaman {page}...")

    try:
        response = requests.get(BASE_URL + str(page), headers=HEADERS)
        if response.status_code != 200:
            print(f"❌ Gagal akses halaman {page}")
            break

        soup = BeautifulSoup(response.content, "html.parser")

        job_cards = soup.find_all("a", {"data-automation": "jobTitle"})

        for job_card in job_cards:
            try:
                title = job_card.get_text(strip=True)
                link = "https://id.jobstreet.com" + job_card["href"]

                # 🔹 LOCATION
                location_tag = job_card.find_next("a", {"data-automation": "jobLocation"})
                location = location_tag.get_text(strip=True) if location_tag else ""

                # 🔹 DETAIL
                requirements, salary, work_type, company = get_job_details(link)

                job_list.append({
                    "Posisi": title,
                    "Perusahaan": company,
                    "Lokasi": location,
                    "Type": work_type,
                    "Gaji": salary,
                    "Requirements": requirements,
                    "Link": link
                })

                print(f"✔ {title}")

                time.sleep(DELAY_DETAIL_PAGE)

            except Exception as e:
                print(f"❌ Error job card: {e}")

        time.sleep(DELAY_LIST_PAGE)

    except Exception as e:
        print(f"❌ Error halaman: {e}")
        break

# =========================
# SAVE DATA (APPEND MODE)
# =========================
df = pd.DataFrame(job_list)

# hapus duplikat dari batch baru
df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

if os.path.exists(FILE_NAME):
    # load data lama
    old_df = pd.read_csv(FILE_NAME)

    # gabungkan
    df = pd.concat([old_df, df], ignore_index=True)

    # hapus duplikat global
    df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

# simpan ulang (clean version)
df.to_csv(FILE_NAME, index=False)

print("\n✅ Selesai! Data berhasil di-update tanpa duplikasi.")


🔄 Scraping halaman 66...
✔ B2C Marketing Officer (Jakarta)
✔ Architect (Revit)
✔ Injection Molding Process Engineer
✔ Staff Akunting Perpajakan
✔ Customer Service / Consultant (Hybrid)
✔ Program Management Trainee (MT) Business Jambi
✔ HR Generalist
✔ OPERATION MANAGER
✔ Section Head Sample
✔ Sales Area
✔ Business Development Assistant Supervisor
✔ Site Acquisition / Expansion Specialist (F&B)
✔ Guest Relation Officer (Villa Management)
✔ Business Development (Outsource Industry)
✔ Marketing Technology
✔ Sales & Commerce Officer
✔ Field Engineer
✔ General Ledger Manager
✔ Sales HORECA (Produk Teh Premium)
✔ Relationship Officer BCA Finance - Sangatta
✔ Supervisor Sipil
✔ Sales Officer (Golf)
✔ Agronomy Assistant
✔ Warehouse Assistant Supervisor
✔ Senior Accountant - F&B
✔ Management Trainee Marketing Branch Tangerang
✔ Finance Accounting & Tax Supervisor (F&B)
✔ Reminder & Desk Collection Staff
✔ Perencana Sipil
✔ ERM (Enterprise Risk Management) Officer
✔ Kitchen Admin Staff
✔ E-Comm

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# =========================
# CONFIG
# =========================
BASE_URL = "https://id.jobstreet.com/id/jobs?page="
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

START_PAGE = 101   # lanjut dari sini
MAX_PAGES = 150   # target halaman akhir

DELAY_LIST_PAGE = 2
DELAY_DETAIL_PAGE = 1

FILE_NAME = "dataset.csv"

job_list = []

# =========================
# FUNCTION: GET DETAIL JOB
# =========================
def get_job_details(job_url):
    try:
        response = requests.get(job_url, headers=HEADERS)
        if response.status_code != 200:
            return "", "", "", ""

        soup = BeautifulSoup(response.content, "html.parser")

        # 🔹 REQUIREMENTS
        combined_requirements = []
        job_ad = soup.find("div", {"data-automation": "jobAdDetails"})

        if job_ad:
            for li in job_ad.find_all("li"):
                text = li.get_text(strip=True)
                if text:
                    combined_requirements.append(text)

        requirements = "; ".join(combined_requirements)

        # 🔹 SALARY
        salary = ""
        salary_tag = soup.find("span", {"data-automation": "job-detail-salary"})
        if salary_tag:
            salary = salary_tag.get_text(strip=True).replace("\xa0", " ")

        # 🔹 WORK TYPE
        work_type = ""
        type_tag = soup.find("span", {"data-automation": "job-detail-work-type"})
        if type_tag:
            a_tag = type_tag.find("a")
            if a_tag:
                work_type = a_tag.get_text(strip=True)

        # 🔹 COMPANY
        company = ""
        company_tag = soup.find("span", {"data-automation": "advertiser-name"})
        if company_tag:
            company = company_tag.get_text(strip=True)

        return requirements, salary, work_type, company

    except Exception as e:
        print(f"❌ Error detail: {e}")
        return "", "", "", ""

# =========================
# MAIN SCRAPING
# =========================
for page in range(START_PAGE, MAX_PAGES + 1):
    print(f"\n🔄 Scraping halaman {page}...")

    try:
        response = requests.get(BASE_URL + str(page), headers=HEADERS)
        if response.status_code != 200:
            print(f"❌ Gagal akses halaman {page}")
            break

        soup = BeautifulSoup(response.content, "html.parser")

        job_cards = soup.find_all("a", {"data-automation": "jobTitle"})

        for job_card in job_cards:
            try:
                title = job_card.get_text(strip=True)
                link = "https://id.jobstreet.com" + job_card["href"]

                # 🔹 LOCATION
                location_tag = job_card.find_next("a", {"data-automation": "jobLocation"})
                location = location_tag.get_text(strip=True) if location_tag else ""

                # 🔹 DETAIL
                requirements, salary, work_type, company = get_job_details(link)

                job_list.append({
                    "Posisi": title,
                    "Perusahaan": company,
                    "Lokasi": location,
                    "Type": work_type,
                    "Gaji": salary,
                    "Requirements": requirements,
                    "Link": link
                })

                print(f"✔ {title}")

                time.sleep(DELAY_DETAIL_PAGE)

            except Exception as e:
                print(f"❌ Error job card: {e}")

        time.sleep(DELAY_LIST_PAGE)

    except Exception as e:
        print(f"❌ Error halaman: {e}")
        break

# =========================
# SAVE DATA (APPEND MODE)
# =========================
df = pd.DataFrame(job_list)

# hapus duplikat dari batch baru
df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

if os.path.exists(FILE_NAME):
    # load data lama
    old_df = pd.read_csv(FILE_NAME)

    # gabungkan
    df = pd.concat([old_df, df], ignore_index=True)

    # hapus duplikat global
    df.drop_duplicates(subset=["Posisi", "Link"], inplace=True)

# simpan ulang (clean version)
df.to_csv(FILE_NAME, index=False)

print("\n✅ Selesai! Data berhasil di-update tanpa duplikasi.")


🔄 Scraping halaman 101...
✔ Marketing Officer (Medan)
✔ SALES REPRESENTATIVE
✔ Apoteker Penanggung Jawab
✔ Project Engineer - Electrical Data Center
✔ Perawat Pelaksana (Penempatan Klinik IK Serang)
✔ LEGAL SENIOR STAFF
✔ Sales Manager Pakan
✔ Data Processor (Remote)
✔ Financial Advisor  - Denpasar & Badung, Mataram, Kupang
✔ QA Customer Service (English)
✔ R&D Assistant Manager
✔ Marketing Digital
✔ Spv Accounting Finance Tax
✔ Fisioterapis (Temporary for Maternity Leave)
✔ Business Analyst & Implementor
✔ Finance and Accounting Manager
✔ SUPERVISOR MARKETING
✔ Direktur Operasional (Civil)
✔ Marketing
✔ Sales Representative
✔ Account Specialist
✔ SALES ENGINEER SURABAYA
✔ Travel Arrangement (Jr. GA Staff)
✔ Deputy Lead Civil Engineer
✔ Production Supervisor
✔ Patient Acces and Engagement Leader
✔ STAFF ACCOUNTING
✔ Content Creator
✔ E-Commerce Specialist
✔ Tour Operation Staff
✔ Remotely Operated Vehicle (ROV) Pilot Technician Trainee
✔ BIM Coordinator

🔄 Scraping halaman 102...
✔ Sr